In [8]:
# camera and audio implmentation

!pip install -q mediapipe==0.10.14 opencv-python-headless librosa

import os
import cv2
import numpy as np
import librosa
import pickle
import mediapipe as mp
from collections import Counter
from tensorflow.keras.models import load_model
from google.colab import files

print("ready")

NameError: name 'audio_classifier' is not defined

In [7]:
# Upload CNN-LSTM
print("1:Upload tennis_cnn_lstm_v1.h5")
uploaded = files.upload()
MODEL_FILE = [f for f in uploaded.keys() if f.endswith('.h5')][0]
model = load_model(MODEL_FILE)
print(f"CNN-LSTM model loaded: {MODEL_FILE}")

# Upload audio SVM
print("\2: Upload Au8dio_svm")
uploaded = files.upload()
PKL_FILE = [f for f in uploaded.keys() if f.endswith('.pkl')][0]
with open(PKL_FILE, "rb") as f:
    audio_model_data = pickle.load(f)
svm_audio    = audio_model_data["svm"]
scaler_audio = audio_model_data["scaler"]
N_MFCC_AUDIO = audio_model_data["n_mfcc"]
CLIP_AUDIO   = audio_model_data["clip_duration"]
print(f"Audio SVM loaded: {PKL_FILE}")

# Upload test video
print("3 : TEST.mp4)")
uploaded = files.upload()
VIDEO_FILE = [f for f in uploaded.keys() if f.endswith('.mp4')][0]
print(f"Video loaded: {VIDEO_FILE}")

1:Upload tennis_cnn_lstm_v1.h5


NameError: name 'files' is not defined

In [ ]:

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.6
)


SELECTED_JOINTS = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27]

cap = cv2.VideoCapture(VIDEO_FILE)
all_skeletons = []
frame_count   = 0

print(f"Extracting skeleton from {VIDEO_FILE}...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb)

    if results.pose_landmarks:
        lm  = results.pose_landmarks.landmark
        pts = [[lm[i].x, lm[i].y] for i in SELECTED_JOINTS]
    else:
        pts = all_skeletons[-1].tolist() if all_skeletons else [[0.0, 0.0]] * len(SELECTED_JOINTS)

    all_skeletons.append(pts)
    frame_count += 1

cap.release()
pose.close()

skeleton = np.array(all_skeletons)
np.save("match_skeleton.npy", skeleton)
print(f"Skeleton extracted → shape: {skeleton.shape} ({frame_count} frames)")

Extracting skeleton from TEST.mp4...
Skeleton extracted → shape: (1003, 11, 2) (1003 frames)


In [ ]:

print("Extracting audio from video...")
os.system(f'ffmpeg -i "{VIDEO_FILE}" -q:a 0 -map a video_audio.wav -y -loglevel quiet')
y_audio, sr_audio   = librosa.load("video_audio.wav", sr=None)
duration_audio      = len(y_audio) / sr_audio
VIDEO_FPS           = cap.get(cv2.CAP_PROP_FPS) if hasattr(cap, 'get') else 61.9


cap_fps = cv2.VideoCapture(VIDEO_FILE)
VIDEO_FPS = cap_fps.get(cv2.CAP_PROP_FPS) or 61.9
cap_fps.release()

print(f"Audio loaded: {sr_audio}Hz | {duration_audio:.1f}s | FPS={VIDEO_FPS:.1f}")


def extract_audio_features(clip, sr):
    if len(clip) < sr * 0.1:
        return None
    try:
        mfcc   = librosa.feature.mfcc(y=clip, sr=sr, n_mfcc=N_MFCC_AUDIO)
        chroma = librosa.feature.chroma_stft(y=clip, sr=sr)
        zcr    = librosa.feature.zero_crossing_rate(clip)
        rmse   = librosa.feature.rms(y=clip)
        return np.concatenate([
            mfcc.mean(axis=1), mfcc.std(axis=1),
            chroma.mean(axis=1),
            zcr.mean(axis=1), rmse.mean(axis=1)
        ])
    except:
        return None


STEP           = 0.1
MIN_GAP        = 1.0
PROB_THRESHOLD = 0.75

audio_detections  = []
last_audio_detect = -MIN_GAP

print("\nRunning AUDIO STANDALONE detector(SVM)")
print("-" * 55)

t = CLIP_AUDIO / 2
while t < duration_audio - CLIP_AUDIO / 2:
    start = max(0, t - 0.1)
    end   = min(duration_audio, t + CLIP_AUDIO)
    clip  = y_audio[int(start * sr_audio):int(end * sr_audio)]

    feat = extract_audio_features(clip, sr_audio)
    if feat is not None:
        prob = svm_audio.predict_proba(scaler_audio.transform([feat]))[0][1]
        if prob >= PROB_THRESHOLD and (t - last_audio_detect) >= MIN_GAP:
            frame_num = int(t * VIDEO_FPS)
            audio_detections.append((t, frame_num, prob))
            last_audio_detect = t
            print(f"  Audio hit: t={t:.2f}s | frame={frame_num:5d} | conf={prob:.3f}")
    t += STEP

print(f"\n AUDIO STANDALONE: {len(audio_detections)} ball hits detected")
print(f"   Timestamps : {[round(d[0],2) for d in audio_detections]}")
print(f"   Frames     : {[d[1] for d in audio_detections]}")


audio_det_array = np.array([[d[0], d[1], d[2]] for d in audio_detections])
np.save("audio_detections.npy", audio_det_array)
print("Saved → audio_detections.npy")


Extracting audio from video...
Audio loaded: 48000Hz | 16.0s | FPS=61.9

Running AUDIO STANDALONE detector(SVM)
-------------------------------------------------------
  Audio hit: t=1.45s | frame=   89 | conf=0.890
  Audio hit: t=2.65s | frame=  164 | conf=0.833
  Audio hit: t=4.05s | frame=  250 | conf=0.849
  Audio hit: t=5.35s | frame=  331 | conf=0.942
  Audio hit: t=6.65s | frame=  411 | conf=0.957
  Audio hit: t=8.25s | frame=  510 | conf=0.759
  Audio hit: t=11.85s | frame=  733 | conf=0.853
  Audio hit: t=12.95s | frame=  801 | conf=0.954
  Audio hit: t=14.25s | frame=  881 | conf=0.929
  Audio hit: t=15.55s | frame=  962 | conf=0.814

 AUDIO STANDALONE: 10 ball hits detected
   Timestamps : [1.45, 2.65, 4.05, 5.35, 6.65, 8.25, 11.85, 12.95, 14.25, 15.55]
   Frames     : [89, 164, 250, 331, 411, 510, 733, 801, 881, 962]
Saved → audio_detections.npy


In [ ]:

# HSV ball detection
def detect_ball_hsv(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    lower_yellow = np.array([25, 120, 140])
    upper_yellow = np.array([40, 255, 255])
    mask = cv2.inRange(hsv, lower_yellow, upper_yellow)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best       = None
    best_score = 0
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 40 or area > 600:
            continue
        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue
        circularity = 4 * np.pi * area / (perimeter ** 2)
        score = area * circularity
        if score > best_score and circularity > 0.65:
            best_score = score
            M = cv2.moments(cnt)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                best = (cx, cy)
    return best


def is_ball_near_player(ball_pos, skel_frame, width, height):
    if ball_pos is None:
        return False
    bx = ball_pos[0] / width
    by = ball_pos[1] / height
    for idx in [0, 1, 2, 3, 4, 5]:
        jx, jy = skel_frame[idx]
        if np.sqrt((bx - jx)**2 + (by - jy)**2) < 0.25:
            return True
    return False


def audio_svm_near(frame_idx, audio_dets, window_frames=45):
    """/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
    Check if any SVM audio detection falls within ±window_frames of frame_idx
    Uses pre-computed audio_detections for speed
    Returns (bool, confidence)
    """
    for t_sec, f_num, conf in audio_dets:
        if abs(int(frame_idx) - int(f_num)) <= window_frames:
            return True, conf
    return False, 0.0

print("Helper functions defined")


Helper functions defined


In [ ]:

THRESHOLD       = 0.82
COOLDOWN_FRAMES = 60
WINDOW_SIZE     = 30
STRIDE          = 5

LABEL_MAP    = {0: "Forehand", 1: "Backhand", 2: "Serve"}
LABEL_COLORS = {0: (0, 200, 80), 1: (200, 80, 0), 2: (0, 80, 255)}


skeleton = np.load("match_skeleton.npy")
print("Skeleton shape:", skeleton.shape)


cap    = cv2.VideoCapture(VIDEO_FILE)
fps    = cap.get(cv2.CAP_PROP_FPS) or 61.9
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

frame_list = []
while cap.isOpened():
    ret, fr = cap.read()
    if not ret: break
    frame_list.append(fr)
cap.release()
print(f"Loaded {len(frame_list)} frames | {fps:.1f} fps | {width}×{height}")


print("Detecting ball positions...")
ball_positions = [detect_ball_hsv(fr) for fr in frame_list]
ball_found     = sum(1 for b in ball_positions if b is not None)
print(f"Ball detected in {ball_found}/{len(frame_list)} frames")


camera_detections    = []
last_detection_frame = -COOLDOWN_FRAMES
buffer               = []

print("\nRunning CAMERA STANDALONE detector (CNN-LSTM + Ball)...")
print("-" * 55)

for i in range(len(skeleton)):
    pts = skeleton[i]
    buffer.append(pts.flatten())
    if len(buffer) > WINDOW_SIZE:
        buffer.pop(0)

    if len(buffer) == WINDOW_SIZE and (i % STRIDE == 0) and (i - last_detection_frame) >= COOLDOWN_FRAMES:
        seq     = np.array(buffer)
        hip_x   = seq[:, 12]
        hip_y   = seq[:, 13]
        seq_3d  = seq.reshape(WINDOW_SIZE, 11, 2)
        hip_3d  = np.stack([hip_x, hip_y], axis=1)[:, None, :]
        seq_3d  = seq_3d - hip_3d
        seq_input = seq_3d.reshape(1, WINDOW_SIZE, 22)

        probs = model.predict(seq_input, verbose=0)[0]
        cls   = int(np.argmax(probs))
        conf  = float(probs[cls])

        if conf >= THRESHOLD:
            ball_near = any(
                is_ball_near_player(ball_positions[j], skeleton[j], width, height)
                for j in range(max(0, i - WINDOW_SIZE), i + 1)
            )
            if ball_near:
                t_sec = i / fps
                camera_detections.append((i, cls, conf, t_sec))
                last_detection_frame = i
                print(f"  Camera hit: frame={i:5d} | t={t_sec:.2f}s | {LABEL_MAP[cls]:8s} | conf={conf:.3f}")

print(f"\n CAMERA STANDALONE: {len(camera_detections)} strokes detected")
f_cam = sum(1 for d in camera_detections if d[1]==0)
b_cam = sum(1 for d in camera_detections if d[1]==1)
s_cam = sum(1 for d in camera_detections if d[1]==2)
print(f"   Forehand: {f_cam} | Backhand: {b_cam} | Serve: {s_cam}")

Skeleton shape: (1003, 11, 2)
Loaded 1003 frames | 61.9 fps | 2340×1080
Detecting ball positions...
Ball detected in 80/1003 frames

Running CAMERA STANDALONE detector (CNN-LSTM + Ball)...
-------------------------------------------------------
  Camera hit: frame=   90 | t=1.45s | Forehand | conf=1.000
  Camera hit: frame=  280 | t=4.52s | Backhand | conf=1.000
  Camera hit: frame=  450 | t=7.27s | Backhand | conf=1.000
  Camera hit: frame=  630 | t=10.18s | Forehand | conf=1.000
  Camera hit: frame=  815 | t=13.17s | Backhand | conf=1.000
  Camera hit: frame=  970 | t=15.67s | Backhand | conf=1.000

 CAMERA STANDALONE: 6 strokes detected
   Forehand: 2 | Backhand: 4 | Serve: 0


In [ ]:

AUDIO_WINDOW_FRAMES = 45


print("Running MULTIMODAL FUSION (timestamp alignment)...")
print("-" * 55)

fused_detections = []
layer_log        = []

for cam_frame, cam_cls, cam_conf, cam_time in camera_detections:

    audio_confirmed, audio_conf = audio_svm_near(
        cam_frame, audio_detections, AUDIO_WINDOW_FRAMES
    )

    fused_detections.append((cam_frame, cam_cls, cam_conf, cam_time, audio_confirmed))
    layer_log.append({
        "frame"     : cam_frame,
        "time_sec"  : cam_time,
        "label"     : LABEL_MAP[cam_cls],
        "cam_conf"  : cam_conf,
        "audio"     : audio_confirmed,
        "audio_conf": audio_conf
    })

    tag = f"CAMERA + AUDIO (audio_conf={audio_conf:.3f})" if audio_confirmed else "CAMERA ONLY "
    print(f"  Frame {cam_frame:5d} | t={cam_time:.2f}s | {LABEL_MAP[cam_cls]:8s} | {tag}")


f_fused       = sum(1 for d in fused_detections if d[1]==0)
b_fused       = sum(1 for d in fused_detections if d[1]==1)
s_fused       = sum(1 for d in fused_detections if d[1]==2)
both_confirmed = sum(1 for d in fused_detections if d[4])
cam_only       = sum(1 for d in fused_detections if not d[4])

print(f"\n{'='*55}")
print(f"FUSION RESULTS")
print(f"  Camera alone detected : {len(camera_detections)} strokes")
print(f"  Audio alone detected  : {len(audio_detections)} hits")
print(f"  Fused (all confirmed) : {len(fused_detections)} strokes")
print(f"  Camera + Audio        : {both_confirmed}")
print(f"  Camera only           : {cam_only}")
print(f"  Forehand : {f_fused}")
print(f"  Backhand : {b_fused}")
print(f"  Serve    : {s_fused}")
print(f"{'='*55}")




output_name = VIDEO_FILE.replace(".mp4", "") + "_multimodal_final.mp4"
out = cv2.VideoWriter(output_name, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))


detection_lookup = {}
for cam_frame, cam_cls, cam_conf, cam_time, audio_ok in fused_detections:
    detection_lookup[cam_frame] = (cam_cls, cam_conf, audio_ok)

print(f"Writing annotated video → {output_name}")

last_det_frame = -1
last_det_info  = None

for i in range(len(frame_list)):
    canvas = frame_list[i].copy()
    pts    = skeleton[i]


    if i in detection_lookup:
        last_det_frame = i
        last_det_info  = detection_lookup[i]


    for j in range(len(pts)):
        x = int(pts[j][0] * width)
        y = int(pts[j][1] * height)
        cv2.circle(canvas, (x, y), 5, (255, 255, 255), -1)


    if ball_positions[i]:
        bx, by = ball_positions[i]
        cv2.circle(canvas, (bx, by), 12, (0, 255, 255), 3)


    audio_active, _ = audio_svm_near(i, audio_detections, AUDIO_WINDOW_FRAMES)
    if audio_active:
        cv2.circle(canvas, (40, height - 40), 15, (0, 255, 255), -1)
        cv2.putText(canvas, "AUDIO CONFIRMED", (62, height - 32),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)


    if last_det_info is not None and last_det_frame <= i < last_det_frame + WINDOW_SIZE:
        det_cls, det_conf, det_audio = last_det_info
        label = LABEL_MAP[det_cls]
        color = LABEL_COLORS[det_cls]


        cv2.putText(canvas, label.upper(), (80, 140),
                    cv2.FONT_HERSHEY_SIMPLEX, 2.8, (0, 0, 0), 10)
        cv2.putText(canvas, label.upper(), (80, 140),
                    cv2.FONT_HERSHEY_SIMPLEX, 2.8, color, 6)


        if det_audio:
            cv2.putText(canvas, "AUDIO CONFIRMED", (80, 200),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 4)
            cv2.putText(canvas, "AUDIO CONFIRMED", (80, 200),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)


    f_c = sum(1 for d in fused_detections if d[1]==0)
    b_c = sum(1 for d in fused_detections if d[1]==1)
    s_c = sum(1 for d in fused_detections if d[1]==2)
    cv2.putText(canvas, f"F:{f_c}  B:{b_c}  S:{s_c}", (width - 320, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3)
    cv2.putText(canvas, f"F:{f_c}  B:{b_c}  S:{s_c}", (width - 320, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 1)

    out.write(canvas)


stats = np.zeros((height, width, 3), dtype=np.uint8)
for text, color, scale, y_pos in [
    ("MULTIMODAL STROKE DETECTION",        (255,255,255), 1.6,  80),
    ("CNN-LSTM  +  Ball HSV  +  Audio SVM",(180,180,180), 1.1, 140),
    (f"Forehand           :  {f_fused}",   (0,200,80),   1.5, 230),
    (f"Backhand           :  {b_fused}",   (200,80,0),   1.5, 300),
    (f"Serve              :  {s_fused}",   (0,80,255),   1.5, 370),
    (f"TOTAL              :  {len(fused_detections)}", (255,255,0), 1.6, 460),
    (f"Camera + Audio     :  {both_confirmed}", (0,255,100), 1.1, 550),
    (f"Camera only        :  {cam_only}",  (0,255,100),  1.1, 610),
    (f"Audio standalone   :  {len(audio_detections)} hits detected", (0,200,255), 1.0, 670),
]:
    cv2.putText(stats, text, (width//2 - 420, y_pos),
                cv2.FONT_HERSHEY_SIMPLEX, scale, color, 3)

for _ in range(int(fps * 4)):
    out.write(stats)

out.release()

print(f"\n Video saved → {output_name}")
files.download(output_name)














Running MULTIMODAL FUSION (timestamp alignment)...
-------------------------------------------------------


NameError: name 'camera_detections' is not defined